In [10]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [11]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-5-nano")

## Runtime Context

In [12]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColourContext
)

In [14]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [15]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='34826fbc-87ea-41b7-ba36-002042e9d875'),
              AIMessage(content='I don’t know your favorite color yet. I don’t have access to your preferences unless you share them.\n\nWant to play a quick guessing game? I’ll guess blue first. Is blue your favorite? If not, tell me a color you like, or answer a couple quick questions and I’ll suggest one.\n\nQuick quiz (optional):\n- Do you prefer cool colors (blue/green/purple) or warm colors (red/orange/yellow)?\n- Do you like bright, vibrant colors or softer, muted tones?\n- Do you lean toward nature-inspired colors (greens/browns) or bold, statement colors (royal blue, magenta, etc.)?\n\nAnswer a few, and I’ll pick a color that fits. Or just tell me some colors you like and I’ll suggest one.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1195, 'prompt_tokens': 12, 'total_tokens':

In [16]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [17]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [18]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='e537cc03-7e84-4872-b38f-a4fe8a09bfb6'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 149, 'total_tokens': 298, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EUXIhulIjck48hlxEhnNawAMat3Uy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0fcbb-b1e6-7951-a004-69993b1924be-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_6mq53GsbnC4AiwIA7dMAOFmE',

## Write to state

In [17]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

In [16]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour,
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [18]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [19]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [20]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='bc1e3055-20cd-4687-b710-83bf30d98ed0'),
              AIMessage(content='I’m doing well, thanks for asking! How about you? If you’d like, you can tell me your favourite colour and I can save it for you. What would you like to do today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 306, 'prompt_tokens': 142, 'total_tokens': 448, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQubDOgiCWptuzZRa3p684kLjtQ2w', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0c93f-e517-7960-95b5

## Read state

In [21]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [22]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='25da739a-84e0-48a8-ba78-82b18f82efca'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 162, 'total_tokens': 445, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQuc2Dr4EI7iXuGDobWRbUSsiP5sr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c940-a8de-7a51-88ce-a6c0b3955786-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_N3iEyVUeWyH6eTDhApnrgxbc', 'ty

In [23]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='25da739a-84e0-48a8-ba78-82b18f82efca'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 162, 'total_tokens': 445, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQuc2Dr4EI7iXuGDobWRbUSsiP5sr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c940-a8de-7a51-88ce-a6c0b3955786-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_N3iEyVUeWyH6eTDhApnrgxbc', 'ty

## Creating subagents

In [24]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [25]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]
)

In [26]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

In [27]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [28]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='b579e6a9-e0b8-4313-9ecd-f3c5140ae068'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2460, 'prompt_tokens': 202, 'total_tokens': 2662, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2432, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQueJwCLAghrnFM0pnX7cdkQVSy72', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c942-c9c5-7832-9c0d-72204c7d603e-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_8BxRSQhoBzyGUh38qPlGPwtp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_me

## Summarize messages

In [21]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [22]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is inquiring about various aspects of a fictional location called Lunapolis on the moon, including its capital, weather, population of cheese miners, and potential union strikes.\n\n## SUMMARY\n\n- The capital of the moon is stated to be Lunapolis.\n- The weather in Lunapolis is characterized by clear skies, with temperatures ranging from a high of 120C to a low of -100C.\n- There are approximately 100,000 cheese miners residing in Lunapolis.\n- It is suggested that the cheese miners' union may go on strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='37514530-fb50-4dd9-9ab2-03076095b7a4'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_

In [23]:
print(response["messages"][-1].content)

If I were Lunapolis’ new president, I would respond to the cheese miners’ union with a calm, transparent, and collaborative approach. Here’s a practical plan I’d follow:

Core principles
- Respect and dignity: Acknowledge the miners’ contributions and right to advocate for their interests.
- Transparency: Share the financial picture and constraints that affect wage and benefit decisions.
- Safety first: Prioritize worker safety in a habitat with extreme temperatures and lunar conditions.
- Shared future: Seek solutions that improve both workers’ welfare and Lunapolis’ long-term stability.

Immediate actions (0–7 days)
- Schedule a formal meeting with union leadership within 48–72 hours to initiate good-faith talks.
- Establish a neutral, time-bound negotiation framework with a clear agenda, ground rules, and a mediator if needed.
- Publish a brief, public “fact sheet” outlining the city’s finances, mining profitability, costs of safety improvements, and the need for sustainable decisio

## Trim/delete messages

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [ ]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

In [24]:
print(response["messages"][-1].content)

If I were Lunapolis’ new president, I would respond to the cheese miners’ union with a calm, transparent, and collaborative approach. Here’s a practical plan I’d follow:

Core principles
- Respect and dignity: Acknowledge the miners’ contributions and right to advocate for their interests.
- Transparency: Share the financial picture and constraints that affect wage and benefit decisions.
- Safety first: Prioritize worker safety in a habitat with extreme temperatures and lunar conditions.
- Shared future: Seek solutions that improve both workers’ welfare and Lunapolis’ long-term stability.

Immediate actions (0–7 days)
- Schedule a formal meeting with union leadership within 48–72 hours to initiate good-faith talks.
- Establish a neutral, time-bound negotiation framework with a clear agenda, ground rules, and a mediator if needed.
- Publish a brief, public “fact sheet” outlining the city’s finances, mining profitability, costs of safety improvements, and the need for sustainable decisio

## Human in the Loop

In [25]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [26]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [27]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [28]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads '
                                                                          'up. '
                                                                          'I’m '
                                                                          'happy '
                                                                          'to '
               

In [29]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John,\n\nNo problem—thanks for the heads up. I’m happy to reschedule. What times tomorrow work for you? If you’d prefer, I can propose a few options.\n\nBest regards,\nSeán'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John,\\n\\nNo problem—thanks for the heads up. I’m happy to reschedule. What times tomorrow work for you? If you’d prefer, I can propose a few options.\\n\\nBest regards,\\nSeán'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='b66d9a9487bc067f1355046abb72737f')]


In [30]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for the heads up. I’m happy to reschedule. What times tomorrow work for you? If you’d prefer, I can propose a few options.

Best regards,
Seán


## Approve

In [31]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='53e1182b-6185-4e88-b7b1-bc916cbc6273'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 467, 'prompt_tokens': 167, 'total_tokens': 634, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EUXlW87gtGjXhIEb8yhgvKYwmXUCu', 'service_tier': 'default', 'finish_reason': 'too

## Reject

In [ ]:
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ),
    config=config # Same thread ID to resume the paused conversation
    )

pprint(response)

In [ ]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## Edit

In [ ]:
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ),
    config=config # Same thread ID to resume the paused conversation
    )

pprint(response)

## Dynamic Models

In [38]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("gpt-6-luna")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)

    return handler(request)

In [33]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [34]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I didn’t water it myself, since I don’t have access to the plant. I can set a daily watering reminder and log when it’s done if you’d like. What time works best for you, and should I note it in the shared plant care log?


In [35]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [39]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

I should clarify: I can’t actually inspect or care for the office plant from here, so I can’t tell how its roots or pot are doing. In general, repot when roots are circling the pot or coming through the drainage holes, or when the soil dries unusually fast. Many houseplants need repotting every 1–2 years; use a pot just one size larger.


In [40]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-6-luna


## Dynamic Prompts

In [41]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [42]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [43]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Dia dhuit! Tá mé go maith, go raibh maith agat. Conas atá tú?


In [44]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

¡Hola! Estoy bien, gracias. ¿Y tú? ¿Qué puedo hacer por ti hoy?


In [45]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

Bonjour ! Je vais très bien, merci. Et vous, comment allez-vous ? Comment puis-je vous aider aujourd’hui ?


## Dynamic Tools

In [51]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [46]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [52]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role

    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools)

    return handler(request)

In [53]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [54]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

275 artists.

Note: The count came from the Artist table:
SELECT COUNT(*) FROM Artist;  -- result: 275
